# Rebuild AD loci × xQTL table — reproducible final workflow

This notebook wraps Alexandre's original integration and Excel-formatting code in a **manifest-driven, isolated run directory**. To add or remove data, edit only the input manifest in Section 1 (or edit the generated CSV). Every run records the selected inputs, parameters, software versions, validation results, and output checksums.

Two modes are supported:

- `full_alexandre`: rebuild method summaries, AD loci, the long AD×xQTL table, and the workbook with Alexandre's driver. Use this when raw/summary inputs changed.
- `workbook_only`: start from an already harmonized long table and regenerate confidence levels, the wide table, and Excel workbook. Use this for fast formatting/filter changes.

> Important: Alexandre's driver expects at least one input for each method block it executes. Removing an entire method family may require disabling that block in a maintained copy of the driver. Removing or adding datasets *within* a retained family is supported by the manifest.

## 1. User configuration — edit this section only

In [ ]:
PROJECT_DIR <- "$AD_LOCI_ROOT"
ALEXANDRE_DIR <- file.path(PROJECT_DIR, "alexandre")
RUN_TAG <- "202609_final"
MODE <- "workbook_only" # full_alexandre | workbook_only
OVERWRITE <- FALSE

# The editable manifest. On first run it is initialized from Alexandre's metadata.
MANIFEST_FILE <- file.path(PROJECT_DIR, "inputs_AD_loci_xQTL_202609_final.csv")
MANIFEST_TEMPLATE <- file.path(PROJECT_DIR, "metadata_analysis_202609.csv")

# Used only in workbook_only mode.
PREBUILT_LONG_TABLE <- file.path(PROJECT_DIR, "xQTL_all_methods_overlap_with_AD_loci_183refresh_202609.csv.gz")

# External resources required by Alexandre's complete driver.
RESOURCE_PATHS <- list(
  gp_coordinates_clean = "/data/gpQTL/gp_coordinates_clean.txt",
  cv2f_score_dir = "/data/analysis_result/cv2f/score/xqtl_feature_max_allcv2f",
  gene_names = "/data/resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list",
  ld_meta_file = "/data/resource/ADSP_R4_EUR_LD/ld_meta_file.tsv",
  gene_gtf = "/data/resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.gtf"
)

PARAM <- list(
  ad_locus_p_threshold = 1e-5,
  genome_wide_p = 5e-8,
  suggestive_p = 1e-6,
  ad_probability_display = 0.1,
  cv2f_top_n = 5L,
  ctwas_pip = 0.75,
  twas_p = 2.5e-6,
  mr_cpip = 0.5,
  mr_min_cs = 2L,
  mr_max_I2 = 0.5
)

### How to add/remove an input

Run the next two cells once. Then edit `inputs_AD_loci_xQTL_202609_final.csv`:

- set `include` to `TRUE` or `FALSE`;
- add a row with the same columns for a new dataset;
- set `Path` relative to `/data`, or use an absolute path;
- keep Alexandre's canonical `Method`, `Modality`, and context naming;
- optionally provide an existing `summary_file` to reuse a harmonized result.

The notebook never modifies the source manifest. It writes a selected, run-local `metadata_analysis.csv`.

In [ ]:
required_packages <- c("data.table", "openxlsx", "stringr", "digest")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages)) stop("Missing R packages: ", paste(missing_packages, collapse = ", "))
suppressPackageStartupMessages({
  library(data.table); library(openxlsx); library(stringr)
})
setDTthreads(0)
dir.create(PROJECT_DIR, recursive = TRUE, showWarnings = FALSE)
setwd(PROJECT_DIR)
stopifnot(MODE %chin% c("full_alexandre", "workbook_only"))

if (!file.exists(MANIFEST_FILE)) {
  stopifnot(file.exists(MANIFEST_TEMPLATE))
  manifest0 <- fread(MANIFEST_TEMPLATE)
  manifest0[, include := TRUE]
  setcolorder(manifest0, c("include", setdiff(names(manifest0), "include")))
  fwrite(manifest0, MANIFEST_FILE)
  message("Created editable manifest: ", MANIFEST_FILE)
}
manifest <- fread(MANIFEST_FILE)
if (!"include" %in% names(manifest)) manifest[, include := TRUE]
manifest[, include := as.logical(include)]
manifest[]

## 2. Validate selected inputs and freeze the run

In [ ]:
required_manifest_cols <- c("context", "Data Type", "Cohort", "Modality", "Method", "Path")
missing_cols <- setdiff(required_manifest_cols, names(manifest))
if (length(missing_cols)) stop("Manifest is missing: ", paste(missing_cols, collapse = ", "))
selected <- manifest[include %in% TRUE]
if (!nrow(selected)) stop("No inputs selected. Set include=TRUE for at least one row.")

resolve_input <- function(x) {
  ifelse(is.na(x) | x == "", NA_character_, ifelse(startsWith(x, "/"), x, file.path("/data", x)))
}
selected[, resolved_path := resolve_input(Path)]
selected[, input_exists := !is.na(resolved_path) & file.exists(resolved_path)]
missing_raw <- selected[!input_exists & (is.na(summary_file) | summary_file == "")]
if (nrow(missing_raw)) {
  print(missing_raw[, .(context, Method, Path, resolved_path)])
  stop("Selected inputs are missing and have no reusable summary_file.")
}

run_dir <- file.path(PROJECT_DIR, paste0("run_", RUN_TAG))
if (dir.exists(run_dir) && !OVERWRITE) stop("Run directory exists: ", run_dir, ". Change RUN_TAG or set OVERWRITE=TRUE.")
dir.create(run_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(run_dir, "outputs"), showWarnings = FALSE)
fwrite(selected[, !c("resolved_path", "input_exists")], file.path(run_dir, "metadata_analysis.csv"))
fwrite(selected[, .(context, Method, Path, resolved_path, input_exists)], file.path(run_dir, "input_audit.tsv"), sep = "\t")
saveRDS(list(config = list(PROJECT_DIR=PROJECT_DIR, ALEXANDRE_DIR=ALEXANDRE_DIR, RUN_TAG=RUN_TAG, MODE=MODE), parameters=PARAM, resources=RESOURCE_PATHS), file.path(run_dir, "run_config.rds"))
cat("Selected rows:", nrow(selected), " | methods:", uniqueN(selected$Method), " | run:", run_dir, "\n")
selected[, .N, by = .(Method, `Data Type`)][order(Method, `Data Type`)]

## 3A. Full Alexandre rebuild

This stages a private copy of Alexandre's scripts and metadata inside the run directory, patches only paths/declared thresholds, and executes the copied driver. Source files under `alexandre/` are not edited.

In [ ]:
if (MODE == "full_alexandre") {
  aux_files <- c("gene_prio_utils.R", "complete_ADlocus_level_summary.R",
                 "contexts_metadata.csv", "columns_metadata.tsv",
                 "excel_metadata.tsv", "pattern_coloring.tsv")
  src <- file.path(ALEXANDRE_DIR, aux_files)
  if (any(!file.exists(src))) stop("Missing Alexandre files: ", paste(aux_files[!file.exists(src)], collapse = ", "))
  stopifnot(all(file.copy(src, run_dir, overwrite = TRUE)))

  driver <- file.path(run_dir, "complete_ADlocus_level_summary.R")
  code <- readLines(driver, warn = FALSE)
  code <- sub("^setwd\\(.*$", sprintf("setwd(%s)", dQuote(run_dir)), code)
  replacements <- c(
    gp_coordinates_clean = RESOURCE_PATHS$gp_coordinates_clean,
    cv2f_score_dir = RESOURCE_PATHS$cv2f_score_dir,
    gene_names = RESOURCE_PATHS$gene_names,
    ld_meta_file = RESOURCE_PATHS$ld_meta_file,
    gene_gtf = RESOURCE_PATHS$gene_gtf
  )
  for (nm in names(replacements)) {
    code <- sub(sprintf("^%s=.*$", nm), sprintf("%s=%s", nm, dQuote(replacements[[nm]])), code)
  }
  # Parameterize the literal thresholds used by the original script.
  code <- gsub("min_pval<1e-5", sprintf("min_pval<%s", format(PARAM$ad_locus_p_threshold, scientific=TRUE)), code, fixed=TRUE)
  code <- gsub("min_pval<5e-8", sprintf("min_pval<%s", format(PARAM$genome_wide_p, scientific=TRUE)), code, fixed=TRUE)
  code <- gsub("susie_pip>0.75", sprintf("susie_pip>%s", PARAM$ctwas_pip), code, fixed=TRUE)
  writeLines(code, driver)
  writeLines(capture.output(sessionInfo()), file.path(run_dir, "sessionInfo_before.txt"))

  old <- getwd(); on.exit(setwd(old), add = TRUE); setwd(run_dir)
  log_file <- file.path(run_dir, paste0("rebuild_", RUN_TAG, ".log"))
  zz <- file(log_file, open = "wt"); sink(zz, split = TRUE); sink(zz, type = "message")
  tryCatch(sys.source(driver, envir = new.env(parent = globalenv())),
           finally = { sink(type = "message"); sink(); close(zz) })
  setwd(old)
  cat("Full rebuild finished. Log:", log_file, "\n")
} else {
  message("Skipped: MODE=workbook_only")
}

## 3B. Select the harmonized long table

In [ ]:
long_candidates <- if (MODE == "full_alexandre") {
  c(file.path(run_dir, "xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz"),
    file.path(run_dir, "xQTL_all_methods_overlap_with_AD_loci_183refresh_202609.csv.gz"))
} else PREBUILT_LONG_TABLE
long_file <- long_candidates[file.exists(long_candidates)][1]
if (is.na(long_file)) stop("No harmonized long table found.")
res_adx <- fread(long_file)
required_long_cols <- c("variant_ID", "ADlocus", "locus_index", "Method", "gene_ID", "context_short", "context_broad2", "qtl_type")
miss <- setdiff(required_long_cols, names(res_adx))
if (length(miss)) stop("Long table is missing: ", paste(miss, collapse=", "))
analysis_loci <- uniqueN(res_adx$ADlocus, na.rm = TRUE)
cat("Loaded", format(nrow(res_adx), big.mark=","), "rows;", analysis_loci, "analysis loci.\n")
res_adx[, .(rows=.N, loci=uniqueN(ADlocus), genes=uniqueN(gene_ID)), by=Method][order(-rows)]

## 4. Recreate Alexandre's confidence levels, wide table, and display subset

In [ ]:
source(file.path(ALEXANDRE_DIR, "gene_prio_utils.R"))
res_scored <- SummarizeTable(copy(res_adx), group.by = "context_short")
res_wide <- WideTable(res_scored, split.by = c("context_broad2", "qtl_type"))

# Faithful to Alexandre's rule, but parameterized. This is not a strict N-variant cap.
res_wide[, top_variants :=
  ((max_variant_inclusion_probability >= PARAM$ad_probability_display) |
     (cV2F_rank <= PARAM$cv2f_top_n)) |
  (max_variant_inclusion_probability_rank == 1 | variant_rank_xqtl == 1 |
     (rank(pval) == 1 & !is.na(pval))),
  by = locus_index]
display <- res_wide[top_variants %in% TRUE & !is.na(locus_index) & variant_ID != ""]
display_loci <- uniqueN(display$ADlocus, na.rm=TRUE)
missing_display_loci <- setdiff(unique(na.omit(res_adx$ADlocus)), unique(na.omit(display$ADlocus)))
cat("Display rows:", nrow(display), " | displayed loci:", display_loci,
    " | analysis loci without a display row:", length(missing_display_loci), "\n")
if (length(missing_display_loci)) print(data.table(ADlocus=missing_display_loci))

## 5. Write the Excel workbook

In [ ]:
columns_file <- file.path(ALEXANDRE_DIR, "columns_metadata.tsv")
excel_file <- file.path(ALEXANDRE_DIR, "excel_metadata.tsv")
colors_file <- file.path(ALEXANDRE_DIR, "pattern_coloring.tsv")
stopifnot(file.exists(columns_file), file.exists(excel_file), file.exists(colors_file))
cols <- fread(columns_file)[keep == 1]
colsmtd <- fread(excel_file)
colorsmtd <- fread(colors_file)
cols <- PrepColsMtd(cols, colsmtd, display)
wb <- CreateExcelFormat(display, columns_mtd = cols, colors = colorsmtd)

for (cont in unique(colsmtd[wildcard == "context_broad2"]$r_name)) {
  block <- res_scored[context_broad2 == cont]
  if (!nrow(block)) next
  block <- SummarizeTable(block, group.by = "qtl_type")
  block <- WideTable(block)
  block[, top_variants :=
    ((max_variant_inclusion_probability >= PARAM$ad_probability_display) | cV2F_rank <= PARAM$cv2f_top_n) |
    (max_variant_inclusion_probability_rank == 1 | variant_rank_xqtl == 1 |
       (rank(pval) == 1 & !is.na(pval))), by = ADlocus]
  block <- block[top_variants %in% TRUE]
  if (!nrow(block)) next
  block[, gwas_significance := fifelse(min_pval < PARAM$genome_wide_p, "genome wide",
                                  fifelse(min_pval < PARAM$suggestive_p, "suggestive", "ns"))]
  block[, mlog10pval := -log10(min_pval)]
  block[, top_confidence := str_extract(xQTL_effects, "CL[0-9]")]
  wb <- CreateExcelFormat(block, columns_mtd=cols, colors=colorsmtd, wb=wb, sheet_name=cont)
  rm(block); gc()
}
xlsx_out <- file.path(run_dir, "outputs", paste0("unified_AD_loci_xQTL_summary_", RUN_TAG, ".xlsx"))
saveWorkbook(wb, xlsx_out, overwrite = TRUE)
cat("Saved:", xlsx_out, "\n")

## 6. Final QA and provenance report

In [ ]:
stopifnot(file.exists(xlsx_out), file.info(xlsx_out)$size > 0)
sheet_names <- getSheetNames(xlsx_out)
xlsx_main <- as.data.table(read.xlsx(xlsx_out, sheet=1, startRow=3))
xlsx_loci <- if ("ADlocus" %in% names(xlsx_main)) uniqueN(xlsx_main$ADlocus, na.rm=TRUE) else NA_integer_

qa <- data.table(
  metric = c("selected_manifest_rows", "selected_methods", "long_rows", "analysis_loci",
             "display_rows_before_export", "display_loci_before_export",
             "xlsx_main_rows", "xlsx_main_loci", "xlsx_sheets", "xlsx_bytes"),
  value = as.character(c(nrow(selected), uniqueN(selected$Method), nrow(res_adx), analysis_loci,
                         nrow(display), display_loci, nrow(xlsx_main), xlsx_loci,
                         length(sheet_names), file.info(xlsx_out)$size))
)
fwrite(qa, file.path(run_dir, "outputs", "QA_summary.tsv"), sep="\t")
fwrite(data.table(ADlocus=missing_display_loci), file.path(run_dir, "outputs", "analysis_loci_without_display_rows.tsv"), sep="\t")
writeLines(capture.output(sessionInfo()), file.path(run_dir, "outputs", "sessionInfo.txt"))
manifest_hash <- digest::digest(file=MANIFEST_FILE, algo="sha256")
xlsx_hash <- digest::digest(file=xlsx_out, algo="sha256")
writeLines(c(paste0("manifest_sha256  ", manifest_hash), paste0("xlsx_sha256  ", xlsx_hash)),
           file.path(run_dir, "outputs", "SHA256SUMS.txt"))
print(qa)
cat("QA PASSED. Analysis loci and displayed loci are reported separately by design.\n")